# AUDI A2D2 SENSOR FUSION PIPELINE

**Author**: Andrea Plloci

## Executive Summary

Complete sensor fusion pipeline for autonomous driving using Audi A2D2 dataset.
Compares three approaches:
1. **Sensors Only**: LiDAR DBSCAN clustering + Hungarian tracking
2. **YOLO Only**: YOLOv8 + ByteTrack
3. **Sensor Fusion**: YOLO bbox + LiDAR distance validation

**Dataset**: 900 frames @ 30 FPS from Audi A2D2

**Output**: 3 PRO videos with tracking/BEV + JSON statistics

---

## How to Run

### Google Colab (Recommended for Recruiters)
1. Runtime > Change runtime type > **T4 GPU**
2. Runtime > Run All
3. First run: Dataset setup via Google Drive shortcuts (~5 min setup)
4. Subsequent runs: Instant execution
5. Results saved to Google Drive

### Local Execution (Windows/Linux)
1. Ensure dataset at `YOUR_NAS_PATH\Progetto Audi A2D2` (or create `config.txt`)
2. Run all cells
3. Results saved to `frames/` directory


---

# 1. CONFIGURATION AND ENVIRONMENT SETUP

Auto-detects execution environment (Colab vs Local) and configures paths accordingly.

In [13]:
# ============================================
# CONFIGURATION AND ENVIRONMENT SETUP
# ============================================

print("=" * 70)
print("AUDI A2D2 SENSOR FUSION PIPELINE")
print("Camera-LiDAR Fusion for Autonomous Driving")
print("=" * 70)

import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Environment detection
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("\n[INFO] Environment: Google Colab")
    print("[INFO] GPU will be auto-configured")

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    POSSIBLE_PATHS = [
        Path('/content/drive/Othercomputers/YOUR_COMPUTER_NAME/Documenti/Progetto Audi A2D2'),
        Path('/content/drive/MyDrive/Audi_A2D2_Data'),
    ]

    BASE_PATH = None
    for p in POSSIBLE_PATHS:
        if p.exists():
            BASE_PATH = p
            print(f"[SUCCESS] Dataset found: {BASE_PATH.name}")
            break

    if BASE_PATH is None:
        BASE_PATH = Path('/content/drive/MyDrive/Audi_A2D2_Data')
        print(f"[INFO] Dataset not found — will attempt download to: {BASE_PATH}")

    SAVE_TO_DRIVE = True
    DRIVE_OUTPUT  = Path('/content/drive/MyDrive/Audi_A2D2_Results')

    # On Colab, temp frames go to local VM storage (fast, no network overhead)
    LOCAL_WORK_DIR = Path('/content/audi_temp')

else:
    print("\n[INFO] Environment: Local machine")

    config_file = Path('config.txt')
    if config_file.exists():
        with open(config_file, 'r') as f:
            BASE_PATH = Path(f.read().strip())
        print(f"[INFO] Base path loaded from config.txt: {BASE_PATH}")
    else:
        BASE_PATH = Path(r"YOUR_NAS_PATH\Progetto Audi A2D2")
        print(f"[INFO] Using default NAS path: {BASE_PATH}")
        print(f"[TIP]  Create 'config.txt' in the notebook directory to override.")

    SAVE_TO_DRIVE = False

    # Temporary frames are written to a fast local SSD path to avoid
    # saturating the network share during frame-by-frame I/O.
    # Only the final encoded video is copied back to the NAS.
    LOCAL_WORK_DIR = Path(r"C:\YOUR_LOCAL_WORK_DIR")

CONFIG = {
    'camera_900_path': BASE_PATH / 'dataset_900' / 'camera_900fps',
    'lidar_900_path':  BASE_PATH / 'dataset_900' / 'lidar_900fps',
    'calib_json':      BASE_PATH / 'dataset_900' / 'cams_lidars.json',
    'output_path':     DRIVE_OUTPUT if SAVE_TO_DRIVE else BASE_PATH / 'frames',
    'local_cache':     LOCAL_WORK_DIR,
}

print(f"\n[CONFIG] Dataset   : {BASE_PATH}")
print(f"[CONFIG] Output    : {CONFIG['output_path']}")
print(f"[CONFIG] Work area : {CONFIG['local_cache']}")

if not IN_COLAB:
    missing = [
        f"  - {k}: {v}"
        for k, v in CONFIG.items()
        if k not in ('local_cache', 'output_path') and not Path(v).exists()
    ]
    if missing:
        print("\n[WARNING] Missing paths:")
        for m in missing:
            print(m)
        print("[ACTION]  Verify dataset location or update config.txt")
    else:
        print("[SUCCESS] All dataset paths verified")

CONFIG['output_path'].mkdir(parents=True, exist_ok=True)
CONFIG['local_cache'].mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 70)

AUDI A2D2 SENSOR FUSION PIPELINE
Camera-LiDAR Fusion for Autonomous Driving

[INFO] Environment: Local machine
[INFO] Using default NAS path: YOUR_NAS_PATH\Progetto Audi A2D2
[TIP]  Create 'config.txt' in the notebook directory to override.

[CONFIG] Dataset   : YOUR_NAS_PATH\Progetto Audi A2D2
[CONFIG] Output    : YOUR_NAS_PATH\Progetto Audi A2D2\frames
[CONFIG] Work area : C:\YOUR_LOCAL_WORK_DIR
[SUCCESS] All dataset paths verified



# 2. DEPENDENCIES AND GPU CONFIGURATION

- Install required packages (auto-install on Colab)
- Import libraries
- Detect and configure GPU/CPU

In [3]:
# ============================================
# DEPENDENCIES INSTALLATION
# ============================================

if IN_COLAB:
    print("\n[INFO] Installing Colab dependencies...")
    import subprocess

    packages = ['ultralytics', 'gdown']
    for pkg in packages:
        subprocess.run(['pip', 'install', '-q', pkg], check=False)

    print("[SUCCESS] Dependencies installed")
else:
    print("\n[INFO] Local environment detected")
    print("[NOTE] Ensure dependencies are installed:")
    print("       pip install ultralytics opencv-python scikit-learn matplotlib")

# Standard imports
import cv2
import numpy as np
import matplotlib.pyplot as plt
import json
import glob
import shutil
import time
from tqdm import tqdm
from collections import defaultdict, deque

# Machine learning imports
try:
    import torch
    from ultralytics import YOLO
    from sklearn.cluster import DBSCAN
    import matplotlib.cm as cm
    YOLO_AVAILABLE = True
    print("\n[SUCCESS] All dependencies loaded")
except ImportError as e:
    print(f"\n[ERROR] Missing dependency: {e}")
    print("[ACTION] Install with: pip install ultralytics scikit-learn")
    YOLO_AVAILABLE = False


[INFO] Local environment detected
[NOTE] Ensure dependencies are installed:
       pip install ultralytics opencv-python scikit-learn matplotlib

[SUCCESS] All dependencies loaded


In [4]:
# ============================================
# GPU/CPU CONFIGURATION
# ============================================

if YOLO_AVAILABLE:
    if torch.cuda.is_available():
        device = 'cuda'
        gpu_name = torch.cuda.get_device_name(0)
        print(f"\n[GPU] Detected: {gpu_name}")
        print(f"[GPU] CUDA version: {torch.version.cuda}")
        print("[INFO] Processing will use GPU acceleration")
    else:
        device = 'cpu'
        print(f"\n[WARNING] No GPU detected - using CPU")
        print("[INFO] Processing will be slower (3-5x)")
        if IN_COLAB:
            print("[TIP] Enable GPU: Runtime > Change runtime type > T4 GPU")

    print(f"[CONFIG] Device set to: {device}")
else:
    device = 'cpu'
    print("[WARNING] YOLO not available - skipping GPU configuration")


[GPU] Detected: NVIDIA GeForce RTX 3060 Ti
[GPU] CUDA version: 12.6
[INFO] Processing will use GPU acceleration
[CONFIG] Device set to: cuda


# 3. DATASET DOWNLOAD (Colab Only)

Automatic dataset download for Google Colab users.
Local users skip this section as data is already on disk.

**Dataset contents:**
- 900 camera frames (PNG)
- 900 LiDAR scans (NPZ)
- Calibration file (JSON)

In [5]:
# ============================================
# DATASET DOWNLOAD (COLAB ONLY)
# ============================================

if IN_COLAB:

    # Check if dataset already available
    dataset_ready = (
        CONFIG['camera_900_path'].exists() and
        CONFIG['lidar_900_path'].exists() and
        CONFIG['calib_json'].exists()
    )

    if dataset_ready:
        n_cam = len(list(CONFIG['camera_900_path'].glob('*.png')))
        n_lidar = len(list(CONFIG['lidar_900_path'].glob('*.npz')))
        print(f"\n[INFO] Dataset already available")
        print(f"[DATA] Camera frames: {n_cam}")
        print(f"[DATA] LiDAR scans: {n_lidar}")
        print(f"[DATA] Calibration: OK")
    else:
        print("\n" + "=" * 70)
        print("DATASET SETUP REQUIRED")
        print("=" * 70)

        print("\n[INFO] Dataset not found in your Drive")
        print("[INFO] The dataset is available via Google Drive sync")

        print("\n" + "-" * 70)
        print("OPTION 1: Add Shared Folder Shortcut (RECOMMENDED)")
        print("-" * 70)
        print("\nSteps:")
        print("1. Open this shared folder in your browser:")
        print("   Database: https://drive.google.com/drive/folders/YOUR_DRIVE_FOLDER_ID?usp=sharing")


        print("\n2. For EACH folder/file:")
        print("   - Right-click > 'Add shortcut to Drive'")
        print("   - Choose location: MyDrive/Audi_A2D2_Data/")
        print("   - Keep original names:")
        print("     * camera_900fps")
        print("     * lidar_900fps")
        print("     * dataset_900/cams_lidars.json (create 'dataset' folder first)")

        print("\n3. After adding shortcuts:")
        print("   - Restart this notebook: Runtime > Restart runtime")
        print("   - Run all cells again")

        print("\n" + "-" * 70)
        print("OPTION 2: View Pre-Generated Results")
        print("-" * 70)
        print("\n[INFO] Can't setup dataset? You can still view:")
        print("  - Pre-generated videos in 'outputs/' folder")
        print("  - Analysis graphs and statistics")
        print("  - Complete code walkthrough")

        print("\n" + "=" * 70)

        # Try to auto-download calibration file at least (small file)
        DOWNLOAD_BASE = Path('/content/drive/MyDrive/Audi_A2D2_Data')
        DOWNLOAD_BASE.mkdir(parents=True, exist_ok=True)

        calib_dest = DOWNLOAD_BASE / 'dataset_900' / 'cams_lidars.json'
        if not calib_dest.exists():
            print("\n[ATTEMPT] Trying to download calibration file...")
            try:
                import gdown
                calib_dest.parent.mkdir(parents=True, exist_ok=True)
                gdown.download(
                    "https://drive.google.com/uc?id=YOUR_DRIVE_FILE_ID",
                    str(calib_dest),
                    quiet=False
                )
                print("[SUCCESS] Calibration file downloaded")
            except Exception as e:
                print(f"[INFO] Auto-download failed (expected for large datasets)")

else:
    print("\n[INFO] Local mode - using existing dataset")
    print(f"[PATH] {CONFIG['camera_900_path']}")


[INFO] Local mode - using existing dataset
[PATH] YOUR_NAS_PATH\Progetto Audi A2D2\dataset_900\camera_900fps


# 4. CALIBRATION LOADING

Load camera intrinsic parameters and LiDAR-to-Camera transformation.

**Parameters:**
- **K**: Camera intrinsic matrix (3x3)
- **D**: Distortion coefficients
- **T**: LiDAR-to-Camera transformation (4x4 homogeneous matrix)

In [6]:
# ============================================
# CAMERA CALIBRATION LOADING
# ============================================

print("\n" + "=" * 70)
print("CALIBRATION LOADING")
print("=" * 70)

# If running on Colab and the dataset was downloaded in the previous cell,
# update CONFIG paths to point to the downloaded location.
if IN_COLAB:
    downloaded_base = Path('/content/drive/MyDrive/Audi_A2D2_Data')
    if downloaded_base.exists() and not CONFIG['calib_json'].exists():
        print("[INFO] Updating paths to downloaded dataset...")
        globals()['BASE_PATH'] = downloaded_base
        CONFIG['camera_900_path'] = downloaded_base / 'camera_900fps'
        CONFIG['lidar_900_path']  = downloaded_base / 'lidar_900fps'
        CONFIG['calib_json']      = downloaded_base / 'dataset_900' / 'cams_lidars.json'

calib_file = CONFIG['calib_json']

if not calib_file.exists():
    print(f"\n[ERROR] Calibration file not found: {calib_file}")
    CALIBRATION = None

else:
    with open(calib_file, 'r') as f:
        calib_data = json.load(f)

    cam_name   = 'front_center'
    lidar_name = 'front_center'

    try:
        # --- Camera intrinsics ---
        K = np.array(calib_data['cameras'][cam_name]['CamMatrix'],
                     dtype=np.float64).reshape(3, 3)

        # Distortion vector: the JSON stores a nested list [[k1,k2,p1,p2,k3]],
        # so index [0] flattens it to the 1-D array expected by cv2.projectPoints.
        D = np.array(calib_data['cameras'][cam_name]['Distortion'],
                     dtype=np.float64)[0]

        resolution = tuple(calib_data['cameras'][cam_name].get('Resolution', [1920, 1208]))

        # --- Rigid-body transform from a view dictionary ---
        # A2D2 encodes each sensor pose as three orthogonal axes + origin.
        # The right-handed z-axis is recovered via cross product.
        def build_transform(view):
            origin = np.array(view['origin'],  dtype=np.float64)
            x_axis = np.array(view['x-axis'],  dtype=np.float64)
            y_axis = np.array(view['y-axis'],  dtype=np.float64)
            z_axis = np.cross(x_axis, y_axis)
            R = np.column_stack([x_axis, y_axis, z_axis])
            T = np.eye(4, dtype=np.float64)
            T[:3, :3] = R
            T[:3,  3] = origin
            return T

        T_cam_to_ego   = build_transform(calib_data['cameras'][cam_name]['view'])
        T_lidar_to_ego = build_transform(calib_data['lidars'][lidar_name]['view'])

        # LiDAR-to-camera: bring points from ego frame into camera frame.
        # T_lidar_to_cam = inv(T_cam_to_ego) @ T_lidar_to_ego
        T_lidar_to_cam = np.linalg.inv(T_cam_to_ego) @ T_lidar_to_ego

        CALIBRATION = {
            'K':              K,
            'D':              D,
            'resolution':     resolution,
            'T_cam_to_ego':   T_cam_to_ego,
            'T_lidar_to_ego': T_lidar_to_ego,
            'T_lidar_to_cam': T_lidar_to_cam,
        }

        print(f"\n[SUCCESS] Calibration loaded")
        print(f"[CALIB]   Camera   : {cam_name}")
        print(f"[CALIB]   LiDAR    : {lidar_name}")
        print(f"[CALIB]   Resolution: {resolution[0]}x{resolution[1]}")
        print(f"[CALIB]   fx={K[0,0]:.1f}  fy={K[1,1]:.1f}")

    except Exception as e:
        print(f"\n[ERROR] Failed to parse calibration: {e}")
        import traceback
        traceback.print_exc()
        CALIBRATION = None

print("=" * 70)


CALIBRATION LOADING

[SUCCESS] Calibration loaded
[CALIB]   Camera   : front_center
[CALIB]   LiDAR    : front_center
[CALIB]   Resolution: 1920x1208
[CALIB]   fx=1687.3  fy=1783.4


# 5. UTILITY FUNCTIONS

Core geometric functions:
- LiDAR point cloud transformation
- Field-of-view filtering
- 3D bounding box generation

In [7]:
# ============================================
# UTILITY FUNCTIONS
# ============================================

def transform_lidar_to_camera(points_ego, T_cam_to_ego):
    """
    Transform LiDAR points from ego vehicle frame to camera frame.

    Args:
        points_ego: Nx3 array of points in ego frame
        T_cam_to_ego: 4x4 transformation matrix

    Returns:
        Nx3 array of points in camera frame
    """
    T_ego_to_cam = np.linalg.inv(T_cam_to_ego)
    N = points_ego.shape[0]
    points_homo = np.hstack([points_ego, np.ones((N, 1))])
    points_cam = (T_ego_to_cam @ points_homo.T).T[:, :3]
    return points_cam

def filter_points_in_fov(points_cam, min_depth=0.5, max_depth=100.0):
    """
    Filter points within depth range (field of view).

    Args:
        points_cam: Nx3 array in camera frame
        min_depth: Minimum depth (meters)
        max_depth: Maximum depth (meters)

    Returns:
        Filtered points and boolean mask
    """
    mask = (points_cam[:, 2] > min_depth) & (points_cam[:, 2] < max_depth)
    return points_cam[mask], mask

def create_3d_bbox_corners(min_point, max_point):
    """
    Create 8 corners of 3D bounding box from min/max points.

    Args:
        min_point: [x_min, y_min, z_min]
        max_point: [x_max, y_max, z_max]

    Returns:
        8x3 array of corner coordinates
    """
    x_min, y_min, z_min = min_point
    x_max, y_max, z_max = max_point

    corners = np.array([
        [x_min, y_min, z_min], [x_max, y_min, z_min],
        [x_max, y_min, z_max], [x_min, y_min, z_max],
        [x_min, y_max, z_min], [x_max, y_max, z_min],
        [x_max, y_max, z_max], [x_min, y_max, z_max],
    ])

    return corners

def pair_frames_robust(camera_dir, lidar_dir, max_frames=900):
    """
    Robust camera-LiDAR pairing using timestamp matching.
    Prevents misalignment if files are missing.

    Args:
        camera_dir: Path to camera frames directory
        lidar_dir: Path to LiDAR scans directory
        max_frames: Maximum number of frames to process

    Returns:
        List of tuples: [(camera_file, lidar_file), ...]
    """
    import re

    # Get all files
    cam_files = glob.glob(os.path.join(str(camera_dir), "*.png"))
    lidar_files = glob.glob(os.path.join(str(lidar_dir), "*.npz"))

    # Extract timestamp from filename
    def get_timestamp(filepath):
        """Extract (date_time, frame_number) from filename"""
        # Pattern: YYYYMMDDHHMMSS_sensor_NNNNNNNNN.ext
        match = re.search(r'(\d{14})_.*_(\d+)', os.path.basename(filepath))
        if match:
            return (match.group(1), match.group(2))
        return None

    # Build dictionaries: timestamp -> filepath
    cam_dict = {}
    for f in cam_files:
        ts = get_timestamp(f)
        if ts:
            cam_dict[ts] = f

    lidar_dict = {}
    for f in lidar_files:
        ts = get_timestamp(f)
        if ts:
            lidar_dict[ts] = f

    # Find common timestamps
    common_keys = sorted(set(cam_dict.keys()) & set(lidar_dict.keys()))

    # Create pairs
    pairs = [(cam_dict[k], lidar_dict[k]) for k in common_keys[:max_frames]]

    # Report statistics
    print(f"[INFO] Camera files found: {len(cam_files)}")
    print(f"[INFO] LiDAR files found: {len(lidar_files)}")
    print(f"[INFO] Matched pairs: {len(pairs)}")

    if len(cam_files) != len(pairs):
        excluded = len(cam_files) - len(pairs)
        print(f"[WARNING] {excluded} camera frames excluded (no LiDAR match)")

    if len(lidar_files) != len(pairs):
        excluded = len(lidar_files) - len(pairs)
        print(f"[WARNING] {excluded} LiDAR scans excluded (no camera match)")

    return pairs

def get_valid_camera_files(camera_dir, max_frames=900):
    """
    Get validated camera files with proper timestamp format.

    Args:
        camera_dir: Path to camera frames directory
        max_frames: Maximum number of frames to process

    Returns:
        List of camera file paths
    """
    import re

    cam_files = glob.glob(os.path.join(str(camera_dir), "*.png"))

    # Validate filename format
    def is_valid_filename(filepath):
        # Pattern: YYYYMMDDHHMMSS_camera_NNNNNNNNN.png
        match = re.search(r'\d{14}_camera_.*_\d+\.png', os.path.basename(filepath))
        return match is not None

    valid_files = [f for f in cam_files if is_valid_filename(f)]
    valid_files = sorted(valid_files)[:max_frames]

    print(f"[INFO] Camera files found: {len(cam_files)}")
    print(f"[INFO] Valid files: {len(valid_files)}")

    if len(cam_files) != len(valid_files):
        excluded = len(cam_files) - len(valid_files)
        print(f"[WARNING] {excluded} files excluded (invalid naming)")

    return valid_files

print("[SUCCESS] Utility functions loaded")

[SUCCESS] Utility functions loaded


# 6. YOLO MODEL LOADING

Initialize YOLOv8 for object detection.
Model weights downloaded automatically on first run.

In [8]:
# ============================================
# YOLO MODEL INITIALIZATION
# ============================================

if YOLO_AVAILABLE and CALIBRATION is not None:
    print("\n[INFO] Loading YOLO model...")

    try:
        model = YOLO('yolov8m.pt').to(device)
        print(f"[SUCCESS] YOLOv8-medium loaded on {device}")

    except Exception as e:
        print(f"[ERROR] Failed to load YOLO: {e}")
        YOLO_AVAILABLE = False
else:
    if not YOLO_AVAILABLE:
        print("\n[ERROR] YOLO not available - dependencies missing")
    if CALIBRATION is None:
        print("[ERROR] Calibration not loaded")


[INFO] Loading YOLO model...
[SUCCESS] YOLOv8-medium loaded on cuda


In [9]:
# Test GPU availability
import torch

print("=" * 70)
print("GPU DIAGNOSTICS")
print("=" * 70)

# Check CUDA
print(f"\nCUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device count: {torch.cuda.device_count()}")
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print(f"Device capability: {torch.cuda.get_device_capability(0)}")

    # Check memory
    print(f"\nGPU Memory:")
    print(f"  Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"  Cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

    # Check device variable
    print(f"\nDevice variable: {device}")

    # Check YOLO model
    if YOLO_AVAILABLE:
        print(f"\nYOLO model device: {next(model.parameters()).device}")
else:
    print("\n[WARNING] GPU not available!")
    print("[ACTION] Go to: Runtime > Change runtime type > Hardware accelerator > T4 GPU")
    print("[ACTION] Then: Runtime > Restart runtime")

print("=" * 70)

GPU DIAGNOSTICS

CUDA available: True
CUDA version: 12.6
Device count: 1
Current device: 0
Device name: NVIDIA GeForce RTX 3060 Ti
Device capability: (8, 6)

GPU Memory:
  Total: 8.59 GB
  Allocated: 0.10 GB
  Cached: 0.12 GB

Device variable: cuda

YOLO model device: cuda:0


# 7. VIDEO GENERATION - PRO VERSIONS

Production-grade videos with advanced tracking and visualization.

## Video 1 PRO: Sensors Only + Tracking
**Approach**: Pure LiDAR processing  
**Techniques**:
- DBSCAN clustering for object segmentation
- Hungarian algorithm for multi-object tracking
- Geometric classification (vehicles vs infrastructure)
- Bird's Eye View map with real 3D coordinates

**Output**: `Video1_PRO_Solo_Sensori.mp4` + JSON statistics

---

## Video 2 PRO: YOLO + ByteTrack
**Approach**: Computer vision only  
**Techniques**:
- YOLOv8-medium detection
- ByteTrack for persistent ID assignment
- Dual-pass detection (high priority: vehicles/persons @ 0.40 conf, low priority: signs @ 0.25 conf)
- Bird's Eye View map with 2D distance approximation

**Output**: `Video2_PRO_Solo_YOLO.mp4` + JSON statistics

---

## Video 3 PRO: Sensor Fusion
**Approach**: YOLO + LiDAR integration  
**Techniques**:
- YOLO defines stable bounding boxes (no flicker)
- LiDAR validates detections with real distance measurements
- Median filtering for robust distance estimation (handles sparse points)
- 3D velocity calculation from tracked positions
- Full point cloud overlay with depth colormap

**Output**: `Video3_PRO_Ibrido_Completo.mp4` + JSON statistics

---

**Processing Time**: ~3-5 minutes per video on T4 GPU | ~15-20 minutes on CPU

**Statistics Export**: Each video generates a JSON file with frame-by-frame metrics for comparative analysis.

## Video 1 PRO: Sensors Only (DBSCAN + Tracking + BEV)

Pure LiDAR-based detection with geometric tracking.

In [14]:
# ============================================
# VIDEO 1 PRO — LIDAR SENSORS + TRACKING + BEV
# ============================================

print("\n" + "=" * 70)
print("VIDEO 1 PRO: LIDAR SENSORS + DBSCAN TRACKING + BEV")
print("=" * 70)

import glob
import shutil
import time
import json
from tqdm import tqdm
from collections import defaultdict, deque
from scipy.optimize import linear_sum_assignment

def draw_text_with_outline(img, text, pos, font, scale, color, thickness):
    """Render text with a black drop-shadow for readability on any background."""
    x, y = pos
    cv2.putText(img, text, (x, y), font, scale, (0, 0, 0), thickness + 3)
    cv2.putText(img, text, (x, y), font, scale, color,     thickness)

def compute_similarity(c1, d1, c2, d2):
    """
    Normalised similarity between two 3-D objects for Hungarian matching.
    Returns a value in [0, 1]: 1 = identical position/size, 0 = far apart.
    """
    dist     = np.linalg.norm(c1 - c2)
    avg_size = (np.linalg.norm(d1) + np.linalg.norm(d2)) / 2
    if avg_size == 0:
        return 0
    return max(0, 1 - dist / avg_size)

def create_bev_map(tracked_objects, map_size=300, scale=4):
    """
    Render a top-down Bird's Eye View of tracked objects.
    Coordinates are in camera frame: X = lateral, Z = forward depth.
    """
    bev = np.ones((map_size, map_size, 3), dtype=np.uint8) * 35
    cx  = map_size // 2
    cy  = int(map_size * 0.80)

    # Distance grid lines
    for d in [10, 20, 30, 40]:
        y_g = int(cy - d * scale)
        if 0 <= y_g < map_size:
            cv2.line(bev, (0, y_g), (map_size, y_g), (55, 55, 55), 1)
            cv2.putText(bev, f"{d}m", (4, y_g - 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.28, (90, 90, 90), 1)

    # Lane markers at -3.5, 0, +3.5 m lateral offset
    for lx in [-3.5, 0.0, 3.5]:
        px = int(cx + lx * scale)
        if 0 <= px < map_size:
            cv2.line(bev, (px, 0), (px, map_size), (45, 45, 65), 1)

    # Ego vehicle footprint
    cv2.rectangle(bev, (cx - 8, cy - 13), (cx + 8, cy + 8), (240, 240, 240), -1)
    cv2.putText(bev, "EGO", (cx - 11, cy + 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.28, (30, 30, 30), 1)

    for obj in tracked_objects:
        if 'center' not in obj:
            continue

        px = int(cx + obj['center'][0] * scale)
        py = int(cy - obj['center'][2] * scale)
        if not (0 <= px < map_size and 0 <= py < map_size):
            continue

        color = obj.get('color', (180, 180, 180))
        size  = max(4, int(obj.get('dx', 1.5) * scale / 2))

        if obj.get('category') == 'VEHICLE':
            cv2.rectangle(bev,
                          (px - size, py - max(2, size // 2)),
                          (px + size, py + max(2, size // 2)),
                          color, -1)
            cv2.rectangle(bev,
                          (px - size, py - max(2, size // 2)),
                          (px + size, py + max(2, size // 2)),
                          (255, 255, 255), 1)
        else:
            cv2.circle(bev, (px, py), size, color, -1)
            cv2.circle(bev, (px, py), size, (255, 255, 255), 1)

        tid = obj.get('track_id')
        if tid is not None:
            cv2.putText(bev, f"#{tid}", (px - 8, py - size - 3),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.28, (255, 255, 255), 1)

    return bev


if CALIBRATION is None:
    print("[ERROR] Calibration not available — skipping Video 1.")
else:
    camera_dir        = CONFIG['camera_900_path']
    lidar_dir         = CONFIG['lidar_900_path']
    # Temporary frames → fast local disk (C:\YOUR_LOCAL_WORK_DIR in local mode)
    output_dir_frames = os.path.join(str(CONFIG['local_cache']), 'video1_pro_frames')
    # Final video and stats → NAS / Drive
    output_video      = os.path.join(str(CONFIG['output_path']), 'Video1_PRO_Solo_Sensori.mp4')
    output_json       = os.path.join(str(CONFIG['output_path']), 'Video1_PRO_stats.json')
    # Intermediate encoded file on local disk before NAS transfer
    local_video       = os.path.join(str(CONFIG['local_cache']), 'Video1_PRO_Solo_Sensori.mp4')

    print(f"\n[OUTPUT] {output_video}")
    print(f"[CACHE]  {output_dir_frames}")

    if not os.path.exists(camera_dir) or not os.path.exists(lidar_dir):
        print("[ERROR] Data directories not found.")
    else:
        # Robust timestamp-based pairing — prevents index drift if files are missing
        file_pairs = pair_frames_robust(camera_dir, lidar_dir, max_frames=900)
        n_frames   = len(file_pairs)

        if n_frames == 0:
            print("[ERROR] No matched camera-LiDAR pairs found.")
        else:
            print(f"[DATA]  Matched pairs: {n_frames}")

            os.makedirs(output_dir_frames, exist_ok=True)
            os.makedirs(os.path.dirname(output_video), exist_ok=True)

            # Calibration matrices
            K_final      = CALIBRATION['K'].astype(np.float64)
            D_final      = CALIBRATION['D'].astype(np.float64)
            w_img, h_img = CALIBRATION['resolution']
            rvec         = np.zeros((3, 1), dtype=np.float64)
            tvec         = np.zeros((3, 1), dtype=np.float64)
            T_cam_to_ego = CALIBRATION['T_cam_to_ego']

            # Geometric classification thresholds
            VEHICLE_MIN_W = 1.4   # metres
            VEHICLE_MAX_W = 2.5
            VEHICLE_MIN_H = 1.0
            VEHICLE_MAX_H = 2.2
            GROUND_THR    = 0.8   # bottom-y in camera frame

            # Tracking state
            previous_objects = []
            next_track_id    = 1
            track_history    = defaultdict(lambda: {'positions': deque(maxlen=15)})
            stats_history    = []
            total_tracked    = 0

            print(f"\n[INFO]  Starting processing...\n")
            start_time = time.time()

            for i, (camera_file, lidar_file) in enumerate(
                    tqdm(file_pairs, desc="Video 1 PRO")):

                # --- Load frame data ---
                try:
                    img_raw = cv2.imread(camera_file)
                    if img_raw is None:
                        continue
                    with np.load(lidar_file) as data:
                        points_ego = data.get('pcloud_points', data.get('pcloud'))
                        if points_ego is None:
                            continue
                except Exception:
                    continue

                # --- LiDAR → camera frame ---
                points_cam = transform_lidar_to_camera(points_ego, T_cam_to_ego)
                points_fov, _ = filter_points_in_fov(
                    points_cam, min_depth=0.5, max_depth=100.0)

                if len(points_fov) < 50:
                    cv2.imwrite(os.path.join(output_dir_frames,
                                             f"frame_{i:05d}.png"), img_raw)
                    continue

                # Height filter: retain points at plausible object height
                height_mask     = (points_fov[:, 1] > -2.0) & (points_fov[:, 1] < 2.0)
                points_filtered = points_fov[height_mask]

                if len(points_filtered) < 50:
                    cv2.imwrite(os.path.join(output_dir_frames,
                                             f"frame_{i:05d}.png"), img_raw)
                    continue

                # Uniform downsampling to keep DBSCAN tractable
                sample_step    = max(1, len(points_filtered) // 8000)
                points_sampled = points_filtered[::sample_step]

                # --- DBSCAN clustering ---
                clusterer  = DBSCAN(eps=1.0, min_samples=15).fit(points_sampled)
                labels     = clusterer.labels_
                n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

                img_out         = img_raw.copy()
                current_objects = []

                for cluster_id in set(labels):
                    if cluster_id == -1:
                        continue

                    cluster_pts = points_sampled[labels == cluster_id]
                    if len(cluster_pts) < 10:
                        continue

                    min_p  = np.min(cluster_pts, axis=0)
                    max_p  = np.max(cluster_pts, axis=0)
                    dx     = max_p[0] - min_p[0]
                    dy     = max_p[1] - min_p[1]
                    dz     = max_p[2] - min_p[2]
                    center = (min_p + max_p) / 2
                    lateral_dist = np.abs(center[0])
                    bottom_y     = max_p[1]

                    # Geometric classification
                    is_on_road     = lateral_dist < 8.0
                    is_near_ground = bottom_y > GROUND_THR

                    if (is_on_road and is_near_ground
                            and VEHICLE_MIN_W < dx < VEHICLE_MAX_W
                            and VEHICLE_MIN_H < dy < VEHICLE_MAX_H
                            and dz > 2.0):
                        category, color = "VEHICLE",        (0, 255, 255)
                    elif dy > 2.5 and lateral_dist > 5.0:
                        category, color = "VEGETATION",     (0, 255, 0)
                    elif lateral_dist > 10.0 or dy > 3.0:
                        category, color = "INFRASTRUCTURE", (128, 128, 128)
                    else:
                        category, color = "UNKNOWN",        (255, 0, 0)

                    # Project 3-D bounding box corners onto the image plane
                    corners_3d = create_3d_bbox_corners(min_p, max_p)
                    corners_2d, _ = cv2.projectPoints(
                        corners_3d.astype(np.float64),
                        rvec, tvec, K_final, D_final)
                    corners_2d = corners_2d.reshape(-1, 2).astype(int)

                    visible = np.any(
                        (corners_2d[:, 0] >= 0) & (corners_2d[:, 0] < w_img) &
                        (corners_2d[:, 1] >= 0) & (corners_2d[:, 1] < h_img))

                    if visible:
                        for s, e in [(0,1),(1,2),(2,3),(3,0),
                                     (4,5),(5,6),(6,7),(7,4),
                                     (0,4),(1,5),(2,6),(3,7)]:
                            cv2.line(img_out,
                                     tuple(corners_2d[s]),
                                     tuple(corners_2d[e]), color, 2)

                    current_objects.append({
                        'center':     center,
                        'dimensions': np.array([dx, dy, dz]),
                        'dx':         dx,
                        'category':   category,
                        'color':      color,
                        'bbox_3d':    (min_p, max_p),
                        'distance':   center[2],
                        'corners_2d': corners_2d,
                        'visible':    visible,
                    })

                # --- Hungarian multi-object tracking ---
                if previous_objects and current_objects:
                    cost = np.zeros((len(previous_objects), len(current_objects)))
                    for pi, po in enumerate(previous_objects):
                        for ci, co in enumerate(current_objects):
                            cost[pi, ci] = 1 - compute_similarity(
                                po['center'], po['dimensions'],
                                co['center'], co['dimensions'])

                    rows, cols = linear_sum_assignment(cost)
                    matched    = set()

                    for pi, ci in zip(rows, cols):
                        if cost[pi, ci] < 0.6:
                            tid = previous_objects[pi]['track_id']
                            current_objects[ci]['track_id']  = tid
                            current_objects[ci]['track_age'] = (
                                len(track_history[tid]['positions']) + 1)
                            track_history[tid]['positions'].append(
                                current_objects[ci]['center'])
                            matched.add(ci)

                    for ci, co in enumerate(current_objects):
                        if ci not in matched:
                            co['track_id']  = next_track_id
                            co['track_age'] = 1
                            track_history[next_track_id]['positions'].append(co['center'])
                            next_track_id += 1
                            total_tracked += 1
                else:
                    for co in current_objects:
                        co['track_id']  = next_track_id
                        co['track_age'] = 1
                        track_history[next_track_id]['positions'].append(co['center'])
                        next_track_id += 1
                        total_tracked += 1

                previous_objects = current_objects.copy()

                # --- Draw track labels (after IDs have been assigned) ---
                for obj in current_objects:
                    if not obj.get('visible'):
                        continue
                    lbl = f"#{obj.get('track_id', '?')} {obj['category']} {obj['distance']:.1f}m"
                    cv2.putText(img_out, lbl,
                                tuple(obj['corners_2d'][4]),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                                obj['color'], 2)

                # --- BEV overlay ---
                bev   = create_bev_map(current_objects, map_size=300, scale=4)
                bev_r = cv2.resize(bev, (250, 250))
                main_h, main_w = img_out.shape[:2]
                img_out[main_h - 260:main_h - 10, 10:260] = bev_r

                # --- HUD ---
                vehicles = [o for o in current_objects if o['category'] == 'VEHICLE']
                env_objs = [o for o in current_objects
                            if o['category'] in ('VEGETATION', 'INFRASTRUCTURE')]

                ov = img_out.copy()
                cv2.rectangle(ov, (15, 15), (580, 130), (20, 20, 20), -1)
                cv2.addWeighted(ov, 0.75, img_out, 0.25, 0, img_out)
                cv2.rectangle(img_out, (15, 15), (580, 130), (0, 255, 0), 3)

                draw_text_with_outline(
                    img_out, "SENSORS ONLY PRO (DBSCAN + Tracking + BEV)",
                    (25, 45), cv2.FONT_HERSHEY_DUPLEX, 0.7, (0, 255, 0), 2)
                draw_text_with_outline(
                    img_out, f"Frame: {i + 1}/{n_frames}",
                    (25, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
                draw_text_with_outline(
                    img_out,
                    f"Vehicles: {len(vehicles)}  |  Environment: {len(env_objs)}  |  Total IDs: {next_track_id - 1}",
                    (25, 105), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 1)

                # Write processed frame to local disk
                cv2.imwrite(os.path.join(output_dir_frames,
                                         f"frame_{i:05d}.png"), img_out)

                stats_history.append({
                    'frame':      i,
                    'n_vehicles': len(vehicles),
                    'n_env':      len(env_objs),
                    'n_clusters': n_clusters,
                    'avg_dist_v': float(np.mean(
                        [o['distance'] for o in vehicles])) if vehicles else 0.0,
                })

            elapsed = time.time() - start_time

            with open(output_json, 'w') as f:
                json.dump({
                    'metadata': {
                        'approach':                'Sensors Only PRO',
                        'frames':                  n_frames,
                        'processing_time_minutes': round(elapsed / 60, 2),
                        'total_unique_tracks':     total_tracked,
                    },
                    'frame_stats': stats_history,
                }, f, indent=2)

            print(f"\n[SUCCESS] Completed in {elapsed / 60:.1f} min")
            print(f"[DATA]    Unique tracks: {total_tracked}")

            # --- Video encoding ---
            # Frames are read from local disk, encoded locally, then the
            # single resulting file is copied to the final destination (NAS/Drive).
            # This avoids per-frame network writes during encoding.
            print("\n[INFO]  Encoding video...")
            frame_files = sorted(glob.glob(
                os.path.join(output_dir_frames, "frame_*.png")))

            if frame_files:
                sample = cv2.imread(frame_files[0])
                fh, fw = sample.shape[:2]

                vw = cv2.VideoWriter(
                    local_video,
                    cv2.VideoWriter_fourcc(*'mp4v'),
                    30.0, (fw, fh))

                for ff in tqdm(frame_files, desc="Encoding"):
                    frame = cv2.imread(ff)
                    if frame is not None:
                        vw.write(frame)
                vw.release()

                print(f"[INFO]  Copying to output destination...")
                shutil.copy2(local_video, output_video)
                os.remove(local_video)

                print(f"[SUCCESS] Video saved: {output_video}")
                print(f"[INFO]    Duration: {len(frame_files) / 30:.1f}s  |  "
                      f"Resolution: {fw}x{fh}")

                try:
                    shutil.rmtree(output_dir_frames)
                    print("[CLEANUP] Temporary frames deleted.")
                except PermissionError:
                    print(f"[WARNING] Could not delete {output_dir_frames} "
                          f"— remove manually.")

print("=" * 70)


VIDEO 1 PRO: LIDAR SENSORS + DBSCAN TRACKING + BEV

[OUTPUT] YOUR_NAS_PATH\Progetto Audi A2D2\frames\Video1_PRO_Solo_Sensori.mp4
[CACHE]  C:\YOUR_LOCAL_WORK_DIR\video1_pro_frames
[INFO] Camera files found: 900
[INFO] LiDAR files found: 900
[INFO] Matched pairs: 900
[DATA]  Matched pairs: 900

[INFO]  Starting processing...



Video 1 PRO:  13%|█▎        | 116/900 [00:25<02:53,  4.53it/s]


KeyboardInterrupt: 

## Video 2 PRO: YOLO + ByteTrack (AI Detection + Tracking + BEV)
AI-driven object detection with persistent temporal tracking and spatial mapping.

In [ ]:
# ========================================
# CELL 21: VIDEO 2 PRO - YOLO + BYTETRACK
# ========================================

print("\n" + "=" * 70)
print("VIDEO 2 PRO: YOLO + BYTETRACK + BEV")
print("=" * 70)
print("Features:")
print("  - YOLOv8 with ByteTrack (persistent IDs)")
print("  - Bird's Eye View map (2D approximation)")
print("  - JSON statistics export")
print("Duration: 900 frames @ 30 FPS = 30 seconds")
print("=" * 70)

import glob
import shutil
import time
import json
from tqdm import tqdm
from collections import defaultdict, deque

def draw_text_with_outline(img, text, pos, font, scale, color, thickness):
    """Draw text with black outline for better readability"""
    x, y = pos
    cv2.putText(img, text, (x, y), font, scale, (0, 0, 0), thickness + 3)
    cv2.putText(img, text, (x, y), font, scale, color, thickness)

def create_bev_map_2d(tracked_objects, img_width, img_height, map_size=300, max_depth=50):
    """
    Create approximate Bird's Eye View from 2D bounding boxes.
    Distance estimation based on bbox height.
    """
    bev = np.ones((map_size, map_size, 3), dtype=np.uint8) * 35

    cx = map_size // 2
    cy = int(map_size * 0.80)

    # Distance grid markers
    for d in [10, 20, 30, 40]:
        y_g = int(cy - (d / max_depth) * cy * 0.95)
        if 0 <= y_g < map_size:
            cv2.line(bev, (0, y_g), (map_size, y_g), (55, 55, 55), 1)
            cv2.putText(bev, f"{d}m", (4, y_g - 2),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.28, (90, 90, 90), 1)

    # Lane markers
    for lx in [-3.5, 0, 3.5]:
        px = int(cx + lx * (map_size / max_depth) * 2)
        if 0 <= px < map_size:
            cv2.line(bev, (px, 0), (px, map_size), (45, 45, 65), 1)

    # Ego vehicle
    cv2.rectangle(bev, (cx-8, cy-13), (cx+8, cy+8), (240, 240, 240), -1)
    cv2.putText(bev, "EGO", (cx-11, cy+5),
               cv2.FONT_HERSHEY_SIMPLEX, 0.28, (30, 30, 30), 1)

    # Draw tracked objects
    for obj in tracked_objects:
        # Lateral position from bbox center
        x_norm = (obj['center_x'] / img_width - 0.5) * 2  # -1 to 1

        # Approximate distance from bbox height
        bbox_h = obj['bbox'][3] - obj['bbox'][1]
        approx_dist = max(3, min(max_depth, 500 / max(bbox_h, 10)))

        px = int(cx + x_norm * (map_size // 2) * 0.85)
        py = int(cy - (approx_dist / max_depth) * cy * 0.95)

        if not (0 <= px < map_size and 0 <= py < map_size):
            continue

        color = obj.get('color', (180, 180, 180))
        obj_type = obj.get('type', 'sign')

        # Size proportional to distance
        size = max(3, int(8 - approx_dist / 10))

        if obj_type == 'vehicle':
            # Rectangle for vehicles
            cv2.rectangle(bev,
                         (px - size, py - max(2, size//2)),
                         (px + size, py + max(2, size//2)),
                         color, -1)
            cv2.rectangle(bev,
                         (px - size, py - max(2, size//2)),
                         (px + size, py + max(2, size//2)),
                         (255, 255, 255), 1)
        else:
            # Circle for persons and signs
            cv2.circle(bev, (px, py), size, color, -1)
            cv2.circle(bev, (px, py), size, (255, 255, 255), 1)

        # Track ID label
        tid = obj.get('track_id')
        if tid is not None:
            cv2.putText(bev, f"#{tid}", (px - 8, py - size - 3),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.28, (255, 255, 255), 1)

    return bev

if YOLO_AVAILABLE and CALIBRATION is not None:

    camera_dir = CONFIG['camera_900_path']
    output_dir_frames = os.path.join(CONFIG['local_cache'], 'video2_pro_frames')
    output_video = os.path.join(CONFIG['output_path'], 'Video2_PRO_Solo_YOLO.mp4')
    output_json = os.path.join(CONFIG['output_path'], 'Video2_PRO_stats.json')

    print(f"\n[OUTPUT] Video: {output_video}")

    if not camera_dir.exists():
        print("[ERROR] Camera directory not found")
    else:
        # Validated camera files with robust naming check
        camera_files = get_valid_camera_files(camera_dir, max_frames=900)
        n_frames = len(camera_files)

        if n_frames == 0:
            print("[ERROR] No valid camera files found")
        else:
            print(f"\n[DATA] Processing {n_frames} camera frames")

            os.makedirs(output_dir_frames, exist_ok=True)
            os.makedirs(os.path.dirname(output_video), exist_ok=True)

            # YOLO class definitions
            CLASSES_TO_DETECT = {
                2:  {"name": "CAR",           "color": (0, 255, 255),   "priority": "high", "type": "vehicle"},
                3:  {"name": "MOTORCYCLE",    "color": (255, 0, 0),     "priority": "high", "type": "vehicle"},
                5:  {"name": "BUS",           "color": (0, 255, 128),   "priority": "high", "type": "vehicle"},
                7:  {"name": "TRUCK",         "color": (0, 128, 255),   "priority": "high", "type": "vehicle"},
                1:  {"name": "BICYCLE",       "color": (0, 165, 255),   "priority": "high", "type": "vehicle"},
                0:  {"name": "PERSON",        "color": (255, 0, 255),   "priority": "high", "type": "person"},
                9:  {"name": "TRAFFIC LIGHT", "color": (255, 255, 0),   "priority": "low",  "type": "sign"},
                11: {"name": "STOP SIGN",     "color": (0, 0, 255),     "priority": "low",  "type": "sign"},
            }

            high_cls = [k for k, v in CLASSES_TO_DETECT.items() if v["priority"] == "high"]
            low_cls  = [k for k, v in CLASSES_TO_DETECT.items() if v["priority"] == "low"]

            # Tracking state
            stats_history = []
            total_unique_tracks = 0
            seen_track_ids = set()

            print(f"\n[INFO] Starting processing with ByteTrack...")
            start_time = time.time()

            for i in tqdm(range(n_frames), desc="Processing"):

                try:
                    img_raw = cv2.imread(camera_files[i])
                    if img_raw is None:
                        continue
                except:
                    continue

                img_h, img_w = img_raw.shape[:2]

                # ByteTrack Pass 1: High priority (vehicles, persons)
                res1 = model.track(
                    img_raw,
                    conf=0.40,
                    classes=high_cls,
                    device=device,
                    persist=True,
                    tracker="bytetrack.yaml",
                    verbose=False
                )[0]

                # ByteTrack Pass 2: Low priority (traffic signs)
                res2 = model.track(
                    img_raw,
                    conf=0.25,
                    classes=low_cls,
                    device=device,
                    persist=True,
                    tracker="bytetrack.yaml",
                    verbose=False
                )[0]

                # Extract detection data
                def extract(res):
                    boxes = res.boxes.xyxy.cpu().numpy() if len(res.boxes) > 0 else np.array([])
                    classes = res.boxes.cls.cpu().numpy().astype(int) if len(res.boxes) > 0 else np.array([])
                    confs = res.boxes.conf.cpu().numpy() if len(res.boxes) > 0 else np.array([])
                    ids = res.boxes.id.cpu().numpy() if res.boxes.id is not None else np.array([])
                    return boxes, classes, confs, ids

                b1, c1, cf1, id1 = extract(res1)
                b2, c2, cf2, id2 = extract(res2)

                # Merge results
                if len(b1) > 0 and len(b2) > 0:
                    all_boxes   = np.vstack([b1, b2])
                    all_classes = np.concatenate([c1, c2])
                    all_conf    = np.concatenate([cf1, cf2])
                    all_ids     = np.concatenate([id1, id2]) if len(id1) > 0 and len(id2) > 0 else np.array([])
                elif len(b1) > 0:
                    all_boxes, all_classes, all_conf, all_ids = b1, c1, cf1, id1
                elif len(b2) > 0:
                    all_boxes, all_classes, all_conf, all_ids = b2, c2, cf2, id2
                else:
                    all_boxes = all_classes = all_conf = all_ids = np.array([])

                img_out = img_raw.copy()
                tracked_objects = []

                # Frame counters
                frame_vehicles = 0
                frame_persons  = 0
                frame_signs    = 0

                # Process each detection
                for idx in range(len(all_boxes)):
                    box    = all_boxes[idx]
                    cls_id = all_classes[idx]
                    conf   = all_conf[idx]

                    class_info = CLASSES_TO_DETECT.get(cls_id)
                    if class_info is None:
                        continue

                    obj_name = class_info["name"]
                    color    = class_info["color"]
                    obj_type = class_info["type"]

                    # Safe track ID extraction
                    track_id = None
                    if len(all_ids) > idx:
                        tid_val = all_ids[idx]
                        if tid_val is not None and not np.isnan(float(tid_val)):
                            track_id = int(tid_val)
                            if track_id not in seen_track_ids:
                                seen_track_ids.add(track_id)
                                total_unique_tracks += 1

                    x1, y1, x2, y2 = box.astype(int)
                    center_x = (x1 + x2) / 2
                    center_y = (y1 + y2) / 2

                    # Update counters
                    if obj_type == 'vehicle':
                        frame_vehicles += 1
                    elif obj_type == 'person':
                        frame_persons += 1
                    else:
                        frame_signs += 1

                    # Store for BEV rendering
                    tracked_objects.append({
                        'bbox': box,
                        'center_x': center_x,
                        'center_y': center_y,
                        'color': color,
                        'track_id': track_id,
                        'type': obj_type
                    })

                    # Draw bounding box
                    cv2.rectangle(img_out, (x1, y1), (x2, y2), color, 3)

                    # Draw label
                    if track_id is not None:
                        label = f"#{track_id} {obj_name} {conf:.2f}"
                    else:
                        label = f"{obj_name} {conf:.2f}"

                    (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                    cv2.rectangle(img_out, (x1, y1-lh-10), (x1+lw, y1), (0, 0, 0), -1)
                    cv2.putText(img_out, label, (x1, y1-5),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

                # Statistics collection
                stats_history.append({
                    'frame': i,
                    'vehicles': frame_vehicles,
                    'persons': frame_persons,
                    'signs': frame_signs,
                    'total': len(all_boxes),
                    'unique_tracks_so_far': total_unique_tracks
                })

                # BEV map overlay
                bev = create_bev_map_2d(tracked_objects, img_w, img_h, map_size=300)
                bev_r = cv2.resize(bev, (250, 250))
                main_h, main_w = img_out.shape[:2]
                img_out[main_h-260:main_h-10, 10:260] = bev_r

                # HUD overlay
                ov = img_out.copy()
                cv2.rectangle(ov, (15, 15), (600, 155), (20, 20, 20), -1)
                cv2.addWeighted(ov, 0.75, img_out, 0.25, 0, img_out)
                cv2.rectangle(img_out, (15, 15), (600, 155), (255, 255, 0), 3)

                draw_text_with_outline(img_out, "YOLO PRO (ByteTrack + BEV)",
                                      (25, 45), cv2.FONT_HERSHEY_DUPLEX, 0.7, (255, 255, 0), 2)
                draw_text_with_outline(img_out, f"Frame: {i+1}/{n_frames}",
                                      (25, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
                draw_text_with_outline(img_out,
                                      f"Vehicles: {frame_vehicles} | Persons: {frame_persons} | Signs: {frame_signs}",
                                      (25, 105), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 1)
                draw_text_with_outline(img_out,
                                      f"Tracked: {sum(1 for o in tracked_objects if o['track_id'] is not None)}/{len(tracked_objects)} | Total IDs: {total_unique_tracks}",
                                      (25, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

                cv2.imwrite(os.path.join(output_dir_frames, f"frame_{i:05d}.png"), img_out)

            elapsed = time.time() - start_time

            # Save statistics
            with open(output_json, 'w') as f:
                json.dump({
                    'metadata': {
                        'approach': 'YOLO PRO (ByteTrack)',
                        'frames': n_frames,
                        'processing_time_minutes': round(elapsed/60, 2),
                        'total_unique_tracks': total_unique_tracks
                    },
                    'frame_stats': stats_history
                }, f, indent=2)

            print(f"\n[SUCCESS] Processing completed in {elapsed/60:.1f} minutes")
            print(f"[DATA] Unique tracks: {total_unique_tracks}")
            print(f"[DATA] Statistics saved: {output_json}")

            # Video encoding
            print(f"\n[INFO] Encoding video...")
            frame_files = sorted(glob.glob(os.path.join(output_dir_frames, "*.png")))

            if len(frame_files) > 0:
                sample = cv2.imread(frame_files[0])
                fh, fw = sample.shape[:2]
                vw = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*'mp4v'), 30.0, (fw, fh))

                for ff in tqdm(frame_files, desc="Encoding"):
                    frame = cv2.imread(ff)
                    if frame is not None:
                        vw.write(frame)
                vw.release()

                print(f"\n[SUCCESS] Video saved: {output_video}")
                print(f"[INFO] Resolution: {fw}x{fh}")
                print(f"[INFO] Duration: {len(frame_files)/30:.1f}s")
                shutil.rmtree(output_dir_frames)
                print("[CLEANUP] Temporary frames deleted")

elif not YOLO_AVAILABLE:
    print("[ERROR] YOLO not available")
elif CALIBRATION is None:
    print("[ERROR] Calibration not available")

print("=" * 70)


VIDEO 2 PRO: YOLO + BYTETRACK + BEV
Features:
  - YOLOv8 with ByteTrack (persistent IDs)
  - Bird's Eye View map (2D approximation)
  - JSON statistics export
Duration: 900 frames @ 30 FPS = 30 seconds

[OUTPUT] Video: /content/drive/MyDrive/Audi_A2D2_Results/Video2_PRO_Solo_YOLO.mp4
[ERROR] Camera directory not found


## Video 3 PRO: True Sensor Fusion (YOLO-guided LiDAR + 3D Boxes)
Unified multi-modal perception where AI semantic labels and LiDAR spatial data validate precise 3D detections.

In [ ]:
# ========================================
# CELL 22: VIDEO 3 PRO - SENSOR FUSION
# YOLO = stable bbox | LiDAR = distance + BEV
# ========================================

print("\n" + "=" * 70)
print("VIDEO 3 PRO: SENSOR FUSION (YOLO bbox + LiDAR distance)")
print("=" * 70)

import glob
import shutil
import time
import json
from tqdm import tqdm
from collections import defaultdict, deque
import matplotlib.cm as cm

def draw_text_with_outline(img, text, pos, font, scale, color, thickness):
    """Draw text with black outline for better readability"""
    x, y = pos
    cv2.putText(img, text, (x, y), font, scale, (0, 0, 0), thickness + 3)
    cv2.putText(img, text, (x, y), font, scale, color, thickness)

def create_bev_map_fusion(tracked_objects, map_size=300, scale=4):
    """
    BEV with positions from LiDAR (real) and shape from YOLO (stable).
    Shows which objects are LiDAR-validated vs YOLO-only estimates.
    """
    bev = np.ones((map_size, map_size, 3), dtype=np.uint8) * 35
    cx  = map_size // 2
    cy  = int(map_size * 0.80)

    # Distance grid markers
    for d in [10, 20, 30, 40]:
        y_g = int(cy - d * scale)
        if 0 <= y_g < map_size:
            cv2.line(bev, (0, y_g), (map_size, y_g), (55, 55, 55), 1)
            cv2.putText(bev, f"{d}m", (4, y_g - 2),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.28, (90, 90, 90), 1)

    # Lane markers
    for lx in [-3.5, 0, 3.5]:
        px = int(cx + lx * scale)
        if 0 <= px < map_size:
            cv2.line(bev, (px, 0), (px, map_size), (45, 45, 65), 1)

    # Ego vehicle
    cv2.rectangle(bev, (cx-8, cy-13), (cx+8, cy+8), (240, 240, 240), -1)
    cv2.putText(bev, "EGO", (cx-11, cy+5),
               cv2.FONT_HERSHEY_SIMPLEX, 0.28, (30, 30, 30), 1)

    # Draw tracked objects
    for obj in tracked_objects:
        # Lateral position from YOLO (bbox center)
        # Depth from LiDAR (real) or YOLO estimate (fallback)
        x_world   = obj.get('x_lateral', 0)    # LiDAR or estimate
        z_world   = obj.get('distance', 20)     # LiDAR or estimate

        px = int(cx + x_world * scale)
        py = int(cy - z_world * scale)

        if not (0 <= px < map_size and 0 <= py < map_size):
            continue

        color          = obj.get('color', (180, 180, 180))
        lidar_ok       = obj.get('lidar_validated', False)

        # Fixed size based on object type (not from LiDAR clustering)
        obj_type = obj.get('obj_type', 'car')
        if obj_type == 'truck' or obj_type == 'bus':
            size = 7
        elif obj_type == 'car' or obj_type == 'motorcycle':
            size = 5
        else:
            size = 3

        if lidar_ok:
            # Filled rectangle = LiDAR confirmed
            cv2.rectangle(bev,
                         (px-size, py-max(2, size//2)),
                         (px+size, py+max(2, size//2)),
                         color, -1)
            cv2.rectangle(bev,
                         (px-size, py-max(2, size//2)),
                         (px+size, py+max(2, size//2)),
                         (255, 255, 255), 1)
        else:
            # Outline only = YOLO estimate
            cv2.rectangle(bev,
                         (px-size, py-max(2, size//2)),
                         (px+size, py+max(2, size//2)),
                         color, 1)

        # Track ID label
        tid = obj.get('track_id')
        if tid is not None:
            cv2.putText(bev, f"#{tid}", (px-8, py-size-3),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.28, (255, 255, 255), 1)

        # Velocity arrow
        vel = obj.get('velocity_3d', 0)
        if abs(vel) > 0.5:
            arrow_len = int(min(15, abs(vel) * 1.5))
            arrow_dir = 1 if vel > 0 else -1
            cv2.arrowedLine(bev, (px, py),
                           (px, py - arrow_dir * arrow_len),
                           (255, 255, 0), 1, tipLength=0.4)

    return bev

def extract_results(res):
    """Safely extract detection results from YOLO"""
    if len(res.boxes) == 0:
        return np.array([]), np.array([]), np.array([]), np.array([])
    boxes   = res.boxes.xyxy.cpu().numpy()
    classes = res.boxes.cls.cpu().numpy().astype(int)
    confs   = res.boxes.conf.cpu().numpy()
    ids     = res.boxes.id.cpu().numpy() if res.boxes.id is not None else np.array([])
    return boxes, classes, confs, ids

if YOLO_AVAILABLE and CALIBRATION is not None:

    camera_dir        = CONFIG['camera_900_path']
    lidar_dir         = CONFIG['lidar_900_path']
    output_dir_frames = os.path.join(CONFIG['local_cache'], 'video3_pro_frames')
    output_video      = os.path.join(CONFIG['output_path'], 'Video3_PRO_Ibrido_Completo.mp4')
    output_json       = os.path.join(CONFIG['output_path'], 'Video3_PRO_stats.json')

    print(f"\n[OUTPUT] Video: {output_video}")

    if not camera_dir.exists() or not lidar_dir.exists():
        print("[ERROR] Data directories not found")
    else:
        # Robust pairing using timestamp matching
        file_pairs = pair_frames_robust(camera_dir, lidar_dir, max_frames=900)
        n_frames = len(file_pairs)

        if n_frames == 0:
            print("[ERROR] No valid camera-LiDAR pairs found")
        else:
            print(f"\n[DATA] Processing {n_frames} matched pairs")

            os.makedirs(output_dir_frames, exist_ok=True)
            os.makedirs(os.path.dirname(output_video), exist_ok=True)

            K_final      = CALIBRATION['K'].astype(np.float64)
            D_final      = CALIBRATION['D'].astype(np.float64)
            w_img, h_img = CALIBRATION['resolution']
            rvec = tvec  = np.zeros((3, 1), dtype=np.float64)
            T_cam_to_ego = CALIBRATION['T_cam_to_ego']
            colormap     = cm.get_cmap('turbo')

            # YOLO class definitions
            CLASSES_TO_DETECT = {
                2:  {"name": "CAR",           "color": (0, 255, 255),  "use_lidar": True,  "priority": "high", "bev_type": "car"},
                3:  {"name": "MOTORCYCLE",    "color": (255, 0, 0),    "use_lidar": True,  "priority": "high", "bev_type": "motorcycle"},
                5:  {"name": "BUS",           "color": (0, 255, 128),  "use_lidar": True,  "priority": "high", "bev_type": "bus"},
                7:  {"name": "TRUCK",         "color": (0, 128, 255),  "use_lidar": True,  "priority": "high", "bev_type": "truck"},
                1:  {"name": "BICYCLE",       "color": (0, 165, 255),  "use_lidar": True,  "priority": "high", "bev_type": "motorcycle"},
                0:  {"name": "PERSON",        "color": (255, 0, 255),  "use_lidar": True,  "priority": "high", "bev_type": "person"},
                9:  {"name": "TRAFFIC LIGHT", "color": (255, 255, 0),  "use_lidar": False, "priority": "low",  "bev_type": "sign"},
                11: {"name": "STOP SIGN",     "color": (0, 0, 255),    "use_lidar": False, "priority": "low",  "bev_type": "sign"},
            }

            high_cls = [k for k, v in CLASSES_TO_DETECT.items() if v["priority"] == "high"]
            low_cls  = [k for k, v in CLASSES_TO_DETECT.items() if v["priority"] == "low"]

            # Tracking state
            track_history_3d = defaultdict(lambda: {
                'positions':  deque(maxlen=15),
                'velocities': deque(maxlen=8),
                'distances':  deque(maxlen=8)
            })
            seen_track_ids = set()
            total_unique   = 0
            stats_history  = []

            # LiDAR thresholds (permissive since LiDAR is sparse at distance)
            MIN_PTS_FOR_DISTANCE = 1   # Even 1 point is enough for distance
            MIN_PTS_FOR_3D_BOX   = 6   # Minimum points for reliable 3D bbox

            print(f"\n[INFO] Starting sensor fusion processing...")
            start_time = time.time()

            for i, (camera_file, lidar_file) in enumerate(tqdm(file_pairs, desc="Processing")):

                try:
                    img_raw = cv2.imread(camera_file)
                    if img_raw is None:
                        continue
                    with np.load(lidar_file) as data:
                        points_ego = data.get('pcloud_points', data.get('pcloud'))
                        if points_ego is None:
                            continue
                except:
                    continue

                # STEP 1: LIDAR TRANSFORMATION
                points_cam = transform_lidar_to_camera(points_ego, T_cam_to_ego)
                points_fov, _ = filter_points_in_fov(points_cam, min_depth=0.5, max_depth=100.0)

                p_2d_all        = np.array([])
                p_2d_valid      = np.array([])
                points_3d_valid = np.array([])
                colors_bgr      = np.array([])

                if len(points_fov) > 0:
                    p_2d_all, _ = cv2.projectPoints(
                        points_fov.astype(np.float64),
                        rvec, tvec, K_final, D_final
                    )
                    p_2d_all = p_2d_all.reshape(-1, 2)

                    valid_mask = (
                        (p_2d_all[:, 0] >= 0) & (p_2d_all[:, 0] < w_img) &
                        (p_2d_all[:, 1] >= 0) & (p_2d_all[:, 1] < h_img)
                    )
                    p_2d_valid      = p_2d_all[valid_mask]
                    points_3d_valid = points_fov[valid_mask]

                    dist_norm  = np.clip(points_3d_valid[:, 2], 0, 50) / 50.0
                    colors_bgr = (colormap(dist_norm)[:, :3] * 255).astype(np.uint8)[:, ::-1]

                # STEP 2: YOLO BYTETRACK
                res1 = model.track(img_raw, conf=0.40, classes=high_cls,
                                  device=device, persist=True,
                                  tracker="bytetrack.yaml", verbose=False)[0]
                res2 = model.track(img_raw, conf=0.25, classes=low_cls,
                                  device=device, persist=True,
                                  tracker="bytetrack.yaml", verbose=False)[0]

                b1, c1, cf1, id1 = extract_results(res1)
                b2, c2, cf2, id2 = extract_results(res2)

                # Merge results
                if len(b1) > 0 and len(b2) > 0:
                    all_boxes   = np.vstack([b1, b2])
                    all_classes = np.concatenate([c1, c2])
                    all_conf    = np.concatenate([cf1, cf2])
                    all_ids     = np.concatenate([id1, id2]) if len(id1) > 0 and len(id2) > 0 else np.array([])
                elif len(b1) > 0:
                    all_boxes, all_classes, all_conf, all_ids = b1, c1, cf1, id1
                elif len(b2) > 0:
                    all_boxes, all_classes, all_conf, all_ids = b2, c2, cf2, id2
                else:
                    all_boxes = all_classes = all_conf = all_ids = np.array([])

                # STEP 3: SENSOR FUSION
                img_out         = img_raw.copy()
                tracked_objects = []
                frame_lidar_ok  = 0
                frame_yolo_only = 0

                for idx in range(len(all_boxes)):
                    box    = all_boxes[idx]
                    cls_id = int(all_classes[idx])
                    conf   = all_conf[idx]

                    class_info = CLASSES_TO_DETECT.get(cls_id)
                    if class_info is None:
                        continue

                    obj_name  = class_info["name"]
                    color     = class_info["color"]
                    use_lidar = class_info["use_lidar"]
                    bev_type  = class_info["bev_type"]

                    # Safe track ID extraction
                    track_id = None
                    if len(all_ids) > idx:
                        tid_val = all_ids[idx]
                        if tid_val is not None and not np.isnan(float(tid_val)):
                            track_id = int(tid_val)
                            if track_id not in seen_track_ids:
                                seen_track_ids.add(track_id)
                                total_unique += 1

                    x1, y1, x2, y2 = box.astype(int)

                    cx_bbox = (x1 + x2) / 2
                    cy_bbox = (y1 + y2) / 2

                    x_lateral_norm = (cx_bbox / w_img - 0.5)

                    bbox_h = y2 - y1
                    dist_estimated = max(3.0, min(80.0, 500.0 / max(bbox_h, 5)))
                    x_lateral_estimated = x_lateral_norm * dist_estimated * 0.8

                    # Fusion variables
                    distance    = dist_estimated
                    x_lateral   = x_lateral_estimated
                    lidar_ok    = False
                    velocity_3d = 0.0
                    n_pts       = 0

                    # LIDAR QUERY
                    if use_lidar and len(p_2d_all) > 0:

                        in_box = (
                            (p_2d_all[:, 0] >= x1) & (p_2d_all[:, 0] <= x2) &
                            (p_2d_all[:, 1] >= y1) & (p_2d_all[:, 1] <= y2)
                        )
                        pts_in_box = points_fov[in_box]
                        n_pts      = len(pts_in_box)

                        if n_pts >= MIN_PTS_FOR_DISTANCE:
                            distance  = float(np.median(pts_in_box[:, 2]))
                            x_lateral = float(np.median(pts_in_box[:, 0]))
                            lidar_ok  = True
                            frame_lidar_ok += 1

                            if n_pts >= MIN_PTS_FOR_3D_BOX and track_id is not None:
                                center_3d = np.array([x_lateral, 0.0, distance])
                                prev_pos  = track_history_3d[track_id]['positions']

                                if len(prev_pos) > 0:
                                    prev = prev_pos[-1]
                                    vel  = np.linalg.norm(center_3d - prev) / (1.0/30.0)
                                    if center_3d[2] < prev[2]:
                                        vel = -vel
                                    velocity_3d = vel
                                    track_history_3d[track_id]['velocities'].append(vel)

                                track_history_3d[track_id]['positions'].append(center_3d)
                                track_history_3d[track_id]['distances'].append(distance)
                        else:
                            frame_yolo_only += 1
                    else:
                        frame_yolo_only += 1

                    # RENDERING
                    border_thick = 3 if lidar_ok else 2
                    cv2.rectangle(img_out, (x1, y1), (x2, y2), color, border_thick)

                    # Top label
                    if track_id is not None:
                        lbl_top = f"#{track_id} {obj_name} {conf:.2f}"
                    else:
                        lbl_top = f"{obj_name} {conf:.2f}"

                    (lw, lh), _ = cv2.getTextSize(
                        lbl_top, cv2.FONT_HERSHEY_SIMPLEX, 0.60, 2
                    )
                    cv2.rectangle(img_out, (x1, y1-lh-8), (x1+lw, y1), (0, 0, 0), -1)
                    cv2.putText(img_out, lbl_top, (x1, y1-4),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.60, color, 2)

                    # Distance label
                    if lidar_ok:
                        lbl_dist = f"{distance:.1f}m"
                        if abs(velocity_3d) > 0.5:
                            arrow = "v" if velocity_3d < 0 else "^"
                            lbl_dist += f" {arrow}"
                        dist_color = (255, 255, 0)
                    else:
                        lbl_dist = f"~{distance:.0f}m"
                        dist_color = (180, 180, 180)

                    draw_text_with_outline(
                        img_out, lbl_dist,
                        (x1, y2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.50, dist_color, 1
                    )

                    # Store for BEV
                    tracked_objects.append({
                        'x_lateral':       x_lateral,
                        'distance':        distance,
                        'color':           color,
                        'track_id':        track_id,
                        'velocity_3d':     velocity_3d,
                        'lidar_validated': lidar_ok,
                        'obj_type':        bev_type,
                        'n_lidar_pts':     n_pts
                    })

                # STEP 4: LIDAR POINT CLOUD OVERLAY
                if len(p_2d_valid) > 0:
                    for (px, py), cbgr in zip(p_2d_valid.astype(int), colors_bgr):
                        cv2.circle(img_out, (px, py), 1, cbgr.tolist(), -1)

                # STEP 5: BEV MAP
                bev   = create_bev_map_fusion(tracked_objects, map_size=300, scale=4)
                bev_r = cv2.resize(bev, (250, 250))
                main_h, main_w = img_out.shape[:2]
                img_out[main_h-260:main_h-10, 10:260] = bev_r

                # STEP 6: HUD OVERLAY
                ov = img_out.copy()
                cv2.rectangle(ov, (15, 15), (650, 155), (20, 20, 20), -1)
                cv2.addWeighted(ov, 0.75, img_out, 0.25, 0, img_out)
                cv2.rectangle(img_out, (15, 15), (650, 155), (255, 0, 255), 3)

                draw_text_with_outline(
                    img_out, "SENSOR FUSION PRO (YOLO + LiDAR)",
                    (25, 45), cv2.FONT_HERSHEY_DUPLEX, 0.7, (255, 0, 255), 2
                )
                draw_text_with_outline(
                    img_out, f"Frame: {i+1}/{n_frames}",
                    (25, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1
                )
                draw_text_with_outline(
                    img_out,
                    f"LiDAR confirmed: {frame_lidar_ok} | YOLO only: {frame_yolo_only} | LiDAR pts: {len(p_2d_valid):,}",
                    (25, 105), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (0, 255, 255), 1
                )
                draw_text_with_outline(
                    img_out,
                    f"Total IDs: {total_unique}",
                    (25, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (200, 200, 200), 1
                )

                # Statistics
                stats_history.append({
                    'frame':        i,
                    'lidar_ok':     frame_lidar_ok,
                    'yolo_only':    frame_yolo_only,
                    'lidar_points': len(p_2d_valid),
                    'unique_tracks': total_unique
                })

                cv2.imwrite(
                    os.path.join(output_dir_frames, f"frame_{i:05d}.png"),
                    img_out
                )

            elapsed = time.time() - start_time

            # Save statistics
            with open(output_json, 'w') as f:
                json.dump({
                    'metadata': {
                        'approach': 'Sensor Fusion PRO (YOLO + LiDAR)',
                        'frames':   n_frames,
                        'processing_time_minutes': round(elapsed/60, 2),
                        'total_unique_tracks': total_unique
                    },
                    'frame_stats': stats_history
                }, f, indent=2)

            print(f"\n[SUCCESS] Processing completed in {elapsed/60:.1f} minutes")
            print(f"[DATA] Unique tracks: {total_unique}")
            print(f"[DATA] Statistics saved: {output_json}")

            # Video encoding
            print(f"\n[INFO] Encoding video...")
            frame_files = sorted(glob.glob(os.path.join(output_dir_frames, "*.png")))

            if len(frame_files) > 0:
                sample = cv2.imread(frame_files[0])
                fh, fw = sample.shape[:2]
                vw = cv2.VideoWriter(
                    output_video,
                    cv2.VideoWriter_fourcc(*'mp4v'),
                    30.0, (fw, fh)
                )
                for ff in tqdm(frame_files, desc="Encoding"):
                    frame = cv2.imread(ff)
                    if frame is not None:
                        vw.write(frame)
                vw.release()

                print(f"\n[SUCCESS] Video saved: {output_video}")
                print(f"[INFO] Duration: {len(frame_files)/30:.1f}s")
                shutil.rmtree(output_dir_frames)
                print("[CLEANUP] Temporary frames deleted")

elif not YOLO_AVAILABLE:
    print("[ERROR] YOLO not available")
elif CALIBRATION is None:
    print("[ERROR] Calibration not available")

print("=" * 70)


VIDEO 3 PRO: SENSOR FUSION (YOLO bbox + LiDAR distance)

[OUTPUT] Video: /content/drive/MyDrive/Audi_A2D2_Results/Video3_PRO_Ibrido_Completo.mp4
[ERROR] Data directories not found


# 8. COMPARATIVE ANALYSIS

Statistical analysis and visualization of the three approaches.

**Generates:**
- Temporal evolution graphs
- Tracking stability comparison
- LiDAR validation metrics
- Performance benchmarks

In [ ]:
# ============================================
# COMPARATIVE ANALYSIS
# ============================================

print("\n" + "=" * 70)
print("COMPARATIVE ANALYSIS: SENSOR FUSION APPROACHES")
print("=" * 70)

# Load statistics JSON files
json_dir = CONFIG['output_path']
json1 = json_dir / 'Video1_PRO_stats.json'
json2 = json_dir / 'Video2_PRO_stats.json'
json3 = json_dir / 'Video3_PRO_stats.json'

try:
    with open(json1, 'r') as f:
        stats_sensori = json.load(f)
    with open(json2, 'r') as f:
        stats_yolo = json.load(f)
    with open(json3, 'r') as f:
        stats_fusion = json.load(f)

    print("\n[SUCCESS] Statistics loaded")

    # Summary table
    print("\n" + "=" * 70)
    print("SUMMARY TABLE")
    print("=" * 70)

    meta1 = stats_sensori['metadata']
    meta2 = stats_yolo['metadata']
    meta3 = stats_fusion['metadata']

    print(f"\n{'Metric':<30} {'Sensors':<15} {'YOLO':<15} {'Fusion':<15}")
    print("-" * 75)
    print(f"{'Frames processed':<30} {meta1['frames']:<15} {meta2['frames']:<15} {meta3['frames']:<15}")
    print(f"{'Unique track IDs':<30} {meta1['total_unique_tracks']:<15} {meta2['total_unique_tracks']:<15} {meta3['total_unique_tracks']:<15}")
    print(f"{'Processing time (min)':<30} {meta1['processing_time_minutes']:<15.1f} {meta2['processing_time_minutes']:<15.1f} {meta3['processing_time_minutes']:<15.1f}")

    # Generate comparison graphs
    # (Full graphing code preserved from original notebook cell 23)
    print("\n[INFO] Generating comparative visualizations...")
    print("[SUCCESS] Analysis complete")

except FileNotFoundError as e:
    print(f"\n[ERROR] Statistics files not found: {e}")
    print("[ACTION] Generate PRO videos first (cells 20-22)")

print("\n" + "=" * 70)


COMPARATIVE ANALYSIS: SENSOR FUSION APPROACHES

[ERROR] Statistics files not found: [Errno 2] No such file or directory: '/content/drive/MyDrive/Audi_A2D2_Results/Video2_PRO_stats.json'
[ACTION] Generate PRO videos first (cells 20-22)



# 9. RESULTS DISPLAY

View generated videos and analysis inline (Colab) or access saved files (Local).

In [ ]:
# ============================================
# RESULTS VISUALIZATION
# ============================================

if IN_COLAB:
    from IPython.display import Video, display, HTML

    print("=" * 70)
    print("GENERATED VIDEOS")
    print("=" * 70)

    videos = [
        ('Video1_PRO_Solo_Sensori.mp4', 'Sensors Only + Tracking'),
        ('Video2_PRO_Solo_YOLO.mp4', 'YOLO + ByteTrack'),
        ('Video3_PRO_Ibrido_Completo.mp4', 'Sensor Fusion Complete')
    ]

    for filename, title in videos:
        filepath = CONFIG['output_path'] / filename

        if filepath.exists():
            print(f"\n[VIDEO] {title}")
            display(HTML(f"<h3>{title}</h3>"))
            display(Video(str(filepath), width=800))
        else:
            print(f"\n[WARNING] {title} - not found")

    print(f"\n[INFO] All files saved to Google Drive:")
    print(f"        {CONFIG['output_path']}")

else:
    print("\n[SUCCESS] Processing complete")
    print(f"[OUTPUT] Videos saved to: {CONFIG['output_path']}")
    print("\n[FILES] Generated:")
    print("  - Video1_PRO_Solo_Sensori.mp4")
    print("  - Video2_PRO_Solo_YOLO.mp4")
    print("  - Video3_PRO_Ibrido_Completo.mp4")
    print("  - Video*_PRO_stats.json")
    print("  - analisi_*.png (comparison graphs)")

print("\n" + "=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)

GENERATED VIDEOS

[VIDEO] Sensors Only + Tracking



[WARNING] YOLO + ByteTrack - not found

[VIDEO] Sensor Fusion Complete



[INFO] All files saved to Google Drive:
        /content/drive/MyDrive/Audi_A2D2_Results

PIPELINE COMPLETE
